In [1]:
import pandas as pd
import numpy as np

In [2]:
path = "/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/"

sp = pd.read_csv(path + "espece.csv")
cc = pd.read_csv(path + "composant_culture.csv")

sdc = pd.read_csv(path + "sdc.csv")[['id','filiere']]
nodeR = pd.read_csv(path + "noeuds_realise.csv")[['id','culture_id','zone_id']]
nodeS = pd.read_csv(path + "noeuds_synthetise.csv")[['id','synthetise_id']]
nodeS_rest = pd.read_csv(path + "noeuds_synthetise_restructure.csv")[['id','culture_id']]
zone = pd.read_csv(path + "zone.csv")[['id','parcelle_id']]
parcelle = pd.read_csv(path + "parcelle.csv")[['id','sdc_id']]
synthetise = pd.read_csv(path + "synthetise.csv")[['id','sdc_id']]
plantation_perenne_realise = pd.read_csv(path + "plantation_perenne_realise.csv")[['id','culture_id','zone_id']]
plantation_perenne_synthetise = pd.read_csv(path + "plantation_perenne_synthetise.csv")[['id','synthetise_id']]
plantation_perenne_synthetise_restructure = pd.read_csv(path + "plantation_perenne_synthetise_restructure.csv")[['id','culture_id']]

/tmp/ipykernel_136855/2638745188.py:11: DtypeWarning: Columns (13,14,30) have mixed types. Specify dtype option on import or set low_memory=False.
  parcelle = pd.read_csv(path + "parcelle.csv")[['id','sdc_id']]


In [3]:
plantation_perenne_synthetise = plantation_perenne_synthetise.merge(plantation_perenne_synthetise_restructure, on='id', how='left')
nodeS = nodeS.merge(nodeS_rest, on='id', how='left')

real = pd.concat([nodeR, plantation_perenne_realise])
syn = pd.concat([nodeS, plantation_perenne_synthetise])

real = real.merge(zone.rename(columns={'id': 'zone_id'}), on='zone_id', how='left')
real = real.merge(parcelle.rename(columns={'id': 'parcelle_id'}), on='parcelle_id', how='left')
syn = syn.merge(synthetise.rename(columns={'id': 'synthetise_id'}), on='synthetise_id', how='left')

df = pd.concat([real, syn])
df = df.merge(sdc.rename(columns={'id': 'sdc_id'}), on='sdc_id', how = 'left')

df = df.loc[df['filiere'].isin(['GRANDES_CULTURES','POLYCULTURE_ELEVAGE'])]
gcpe_culture = df.groupby('culture_id').size().reset_index(name='count')

In [ ]:
with open('/home/administrateur/Bureau/' + "liste_culture_id_gcpe.txt", "w") as f:
    for item in gcpe_culture['culture_id'].unique().tolist():
        f.write(f"{item}\n")

In [ ]:
gcpe_cc = cc.merge(gcpe_culture, on='culture_id', how='inner')
gcpe_cc = gcpe_cc.groupby('espece_id').agg({'count':'sum'}).reset_index()

sp_gcpe = sp.merge(gcpe_cc.rename(columns={'espece_id': 'id'}), on='id', how='inner')
sp_gcpe = sp_gcpe.loc[sp_gcpe['count'] > 0].sort_values('count', ascending=False)

In [ ]:
# # Préparer un premier jet de typologie d'espece
# sp_gcpe['new'] = sp_gcpe['libelle_espece_botanique'].str.cat([sp_gcpe['libelle_qualifiant_aee'], sp_gcpe['libelle_type_saisonnier_aee'], sp_gcpe['libelle_destination_aee']], sep=' ', na_rep='')
# sp_gcpe['new'] = sp_gcpe['new'].str.replace(r'\s+', ' ', regex=True).str.strip()

# # sp_gcpe.to_csv('/home/administrateur/Bureau/' + "ref_espece_gcpe.csv", index=False)